Extract Item7, MD&A, from your 100 10-Ks (/content/drive/MyDrive/Fall2026TextMining_me/Lanqing10Ks). Put MD&A in your dataframe in one column. Run all of your code on the MD&A and put in dataframe. (Your df will contain data for the entire 10-K, and data for the MD&A.)

Save your df as a pickle object so you don't have to start from scratch any more!

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# This is the hardest homework ever :(

Pickle example:

"""
for each file:
  new_row = pd.DataFrame({'filename': [filename], 'companyName': [companyName], 'CIK': [CIK], 'uploadDate': [uploadDate]})
  df = pd.concat([df, new_row], ignore_index=True)

def getFogScore(text):
  fogscore = ...
  return fogScore

df['fogScore'] = df['cleanText'].apply(getFogScore)

after dataFrame is created, after you have cleaned your text

df.to_pickle(somePathinYourGoogleDrive/dataFrame.pkl)

"""

In [ ]:
import re

testString1 = "blah blah blah Item 7.    Management's Discussion and Analysis of Financial Condition and Results of Operations The following discussion should be read..."
testString2 = testString1 + "Item 7A.    Quantitative and Qualitative Disclosures About Market Risk"

item7RegEx = re.compile("Item 7")
print(item7RegEx.findall(testString2))

item7RegEx2 = re.compile("Item 7.+Item 7A")
print(item7RegEx2.findall(testString2))

item7RegEx3 = re.compile("Item 7.+?Item 7A")
print(item7RegEx3.findall(testString2))

item7RegEx4 = re.compile("Item 7(.+?)Item 7A", re.S)
print(item7RegEx4.search(testString2).group(1))

item7RegEx5 = re.compile("I(tem 7(.+?)Item 7A)", re.S)
print(item7RegEx5.search(testString2).group(1))
print(item7RegEx5.search(testString2).group(2))

['Item 7', 'Item 7']
["Item 7.    Management's Discussion and Analysis of Financial Condition and Results of Operations The following discussion should be read...Item 7A"]
["Item 7.    Management's Discussion and Analysis of Financial Condition and Results of Operations The following discussion should be read...Item 7A"]
.    Management's Discussion and Analysis of Financial Condition and Results of Operations The following discussion should be read...
tem 7.    Management's Discussion and Analysis of Financial Condition and Results of Operations The following discussion should be read...Item 7A
.    Management's Discussion and Analysis of Financial Condition and Results of Operations The following discussion should be read...


In [ ]:
import nltk
nltk.download('punkt_tab')

import os
import re
import glob
import pandas as pd
import html
from html.parser import HTMLParser
from nltk.tokenize import sent_tokenize, word_tokenize

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


# Syllable Count same as before

In [ ]:
CMUDICT_PATH = "/content/drive/MyDrive/Fall 2026 Text Mining/week 04/cmudict.txt"

cmudict = {}
with open(CMUDICT_PATH, encoding="latin-1") as f:
  for line in f:
    if line.startswith(";;;"):
      continue
    cols = line.split()
    if not cols:
      continue
    word = cols[0]
    if "(" in word:
      word = word[:word.index("(")]
    cmudict.setdefault(word.lower(), cols[1:])

print(len(cmudict))

125764


In [ ]:
def countVowelGroups(word):
  word = word.lower()
  groups = re.findall(r"[aeiouy]+", word)
  numSyllables = len(groups)
  if word.endswith("e") and not word.endswith(("le", "ee", "ye")) and numSyllables > 1:
    numSyllables -= 1
  return max(numSyllables, 1)

syllableCache = {}

def getSyllables(words):
  word = words.lower()
  if word in syllableCache:
    return syllableCache[word]

  phonemes = cmudict.get(word)
  if phonemes is None:
    syllables = countVowelGroups(word)
  else:
    numSyllables = sum(1 for p in phonemes if p[-1].isdigit())
    syllables = numSyllables

  syllableCache[word] = syllables
  return syllables

def isComplexWord(word):
  return getSyllables(word) > 2

print(getSyllables("Kevin"), isComplexWord("Kevin"))
print(getSyllables("readability"), isComplexWord("readability"))

2 False
5 True


# Future Sentences same as before

In [ ]:
FUTURE_WORDS = {"will", "shall", "may", "might", "should"}

def isFuture(sentence, words=None):
  if words is None:
    words = [w.lower() for w in word_tokenize(sentence)]

  for word in words:
    if word in FUTURE_WORDS:
      return True

  # "going to" is two tokens; NLTK also splits "gonna" into "gon" + "na"
  for i in range(len(words) - 1):
    if words[i] == "going" and words[i + 1] == "to":
      return True
    if words[i] == "gon" and words[i + 1] == "na":
      return True

  return False


text = "NLTK is gonna be a great NLP toolkit. It makes processing text easy! Revenue will grow."
for s in sent_tokenize(text):
  print(isFuture(s), "|", s)

True | NLTK is gonna be a great NLP toolkit.
False | It makes processing text easy!
True | Revenue will grow.


# Sentiment Words same as before

In [ ]:
lm = pd.read_csv("/content/drive/MyDrive/Fall 2026 Text Mining/week 03/Lanqing Jia/Loughran-McDonald_MasterDictionary_1993-2025.csv")
lm["Word"] = lm["Word"].astype(str).str.lower()

positive  = set(lm.loc[lm["Positive"]  > 0, "Word"])
negative  = set(lm.loc[lm["Negative"]  > 0, "Word"])
litigious = set(lm.loc[lm["Litigious"] > 0, "Word"])

print(len(positive), len(negative), len(litigious))

347 2345 903


In [ ]:
WORD_RE = re.compile(r"^[a-z][a-z']*$")

def analyzeDocument(text):
  stats = {
      "sentenceCount": 0, "future_sentenceCount": 0,
      "wordCount": 0,     "future_wordCount": 0,
      "complexWordCount": 0, "future_complexWordCount": 0,
      "syllableCount": 0, "future_syllableCount": 0,
      "PosWordCount": 0,  "future_PosWordCount": 0,
      "NegWordCount": 0,  "future_NegWordCount": 0,
      "LitWordCount": 0,  "future_LitWordCount": 0,
  }

  for sentence in sent_tokenize(text):
    tokens = [w.lower() for w in word_tokenize(sentence)]
    words = [w for w in tokens if WORD_RE.match(w)]
    if not words:
      continue

    future = isFuture(sentence, tokens)

    stats["sentenceCount"] += 1
    if future:
      stats["future_sentenceCount"] += 1

    for word in words:
      syl = getSyllables(word)
      complexWord = syl > 2

      stats["wordCount"] += 1
      stats["syllableCount"] += syl
      if complexWord:
        stats["complexWordCount"] += 1
      if word in positive:
        stats["PosWordCount"] += 1
      elif word in negative:
        stats["NegWordCount"] += 1
      elif word in litigious:
        stats["LitWordCount"] += 1

      if future:
        stats["future_wordCount"] += 1
        stats["future_syllableCount"] += syl
        if complexWord:
          stats["future_complexWordCount"] += 1
        if word in positive:
          stats["future_PosWordCount"] += 1
        elif word in negative:
          stats["future_NegWordCount"] += 1
        elif word in litigious:
          stats["future_LitWordCount"] += 1

  return stats

In [ ]:
def fogIndex(wordCount, sentenceCount, complexWordCount):
  if wordCount == 0 or sentenceCount == 0:
    return 0.0
  return 0.4 * ((wordCount / sentenceCount) + 100.0 * (complexWordCount / wordCount))

def rate(numerator, denominator):
  if denominator == 0:
    return 0.0
  return numerator / denominator

def makeColumns(s, prefix=""):
  cols = {}
  for who in ["", "future_"]:
    wordCount        = s[who + "wordCount"]
    sentenceCount    = s[who + "sentenceCount"]
    complexWordCount = s[who + "complexWordCount"]
    syllableCount    = s[who + "syllableCount"]

    cols[prefix + who + "wordCount"]        = wordCount
    cols[prefix + who + "sentenceCount"]    = sentenceCount
    cols[prefix + who + "complexWordCount"] = complexWordCount
    cols[prefix + who + "PosWordCount"]     = s[who + "PosWordCount"]
    cols[prefix + who + "NegWordCount"]     = s[who + "NegWordCount"]
    cols[prefix + who + "LitWordCount"]     = s[who + "LitWordCount"]
    cols[prefix + who + "PosRate"]          = rate(s[who + "PosWordCount"], wordCount)
    cols[prefix + who + "NegRate"]          = rate(s[who + "NegWordCount"], wordCount)
    cols[prefix + who + "LitRate"]          = rate(s[who + "LitWordCount"], wordCount)
    cols[prefix + who + "complexRate"]      = rate(complexWordCount, wordCount)
    cols[prefix + who + "wordsPerSentence"] = rate(wordCount, sentenceCount)
    cols[prefix + who + "syllablesPerWord"] = rate(syllableCount, wordCount)
    cols[prefix + who + "FOG"]              = fogIndex(wordCount, sentenceCount, complexWordCount)

  cols[prefix + "futureSentenceRate"] = rate(s["future_sentenceCount"], s["sentenceCount"])
  cols[prefix + "futureWordRate"]     = rate(s["future_wordCount"], s["wordCount"])
  return cols

# NEW! Extracting Item7

KNOWLEDGE: each txt file is more than one doc, but a whole EDGAR submission with complex layout, only the first is the 10-K

In [ ]:
# Step 1: SEC submission file  ->  plain text of the filing itself

# we want only the first filing block <DOCUMENT>

def firstDocument(submission):
  block = re.search(r"<DOCUMENT>(.*?)</DOCUMENT>", submission, re.I | re.S)
  return block.group(1) if block else "" # for those with no <DOCUMENT> block at all (e.g. file 14)

def getFormType(submission):
  """There could be 'NT 10-K' for a late-filing notice, or '' if unreadable, we identify them"""
  match = re.search(r"<TYPE>[ \t]*([^\r\n<]+)", firstDocument(submission), re.I)
  # Why [ \t]* not \s*? \s* skips new lines as well while [ \t]* skips spaces/tabs only
  # [^\r\n<]+ -- one or more characters that are not a newline and not "<" -- read to the end of the line and stop early if another tag begins
  return match.group(1).strip() if match else ""

def getDocumentHtml(submission):
  """Drops the <TYPE><SEQUENCE><FILENAME> header lines and keeps only the HTML part"""
  block = firstDocument(submission)
  match = re.search(r"<TEXT>(.*?)</TEXT>", block, re.I | re.S)
  return match.group(1) if match else block # fallback if a block has no <TEXT> tag

class SECTextExtractor(HTMLParser):
  """HTML basics: block tags start a new visual line,
  while for skip tags, skipDepth counts how deep you are inside one of those tags,
  and handle_data throws text away while the count is above zero"""

  BLOCK_TAGS = {
      "p", "div", "br", "tr", "td", "th", "li", "table", "section", "article",
      "h1", "h2", "h3", "h4", "h5", "h6"
  }
  SKIP_TAGS = {"script", "style", "noscript", "head", "ix:hidden"}
  # ix:hidden is specific to SEC filings: it is the Inline-XBRL block holding machine-readable tags that are invisible to a human reader

  def __init__(self):
    super().__init__(convert_charrefs=True)
    self.parts = []
    self.skipDepth = 0

  def handle_starttag(self, tag, attrs):
    tag = tag.lower()
    if tag in self.SKIP_TAGS:
      self.skipDepth += 1
    elif self.skipDepth == 0 and tag in self.BLOCK_TAGS:
      self.parts.append("\n")

  def handle_endtag(self, tag):
    tag = tag.lower()
    if tag in self.SKIP_TAGS:
      self.skipDepth = max(0, self.skipDepth - 1)
    elif self.skipDepth == 0 and tag in self.BLOCK_TAGS:
      self.parts.append("\n")

  def handle_data(self, data):
    if self.skipDepth == 0:
      self.parts.append(data)

def cleanSECText(submission):
  parser = SECTextExtractor()
  parser.feed(getDocumentHtml(submission))
  parser.close()

  text = html.unescape("".join(parser.parts))      # second pass: catches double-escaped entities
  text = text.replace("\xa0", " ").replace("\u200b", "").replace("\u2028", "\n")
  text = re.sub(r"[^\S\n]+", " ", text)            # any run of spaces/tabs -> one space
  text = re.sub(r" *\n *", "\n", text)             # no spaces hugging a line break
  text = re.sub(r"\n{3,}", "\n\n", text)           # at most one blank line
  return text.strip()

In [ ]:
# Step 2: two patterns -- where Item 7 starts, and where it ends

# A real heading starts its own line (^ with re.M).
# we want to identify false situations like "...as discussed in Item 7 below" where "Item 7" appears mid-sentence

# GAP exists because many filers put "Item" and "7" in separate HTML cells, so the cleaned text reads "ITEM\n7." per checking, 7/100 filings do this

GAP   = r"[ \t]*\n?[ \t]*"            # spaces, and at most one line break
PUNCT = r"[.:)\-\u2013\u2014]?"       # the separator, if the filer used one
                                      # (\u2013 = en dash, \u2014 = em dash

ITEM7_RE  = re.compile(r"^[ \t]*item" + GAP + r"7" + GAP + PUNCT + r"\s{0,3}management",
                       re.I | re.M)
ITEM7A_RE = re.compile(r"^[ \t]*item" + GAP + r"(?:7A|8)(?![0-9A-Za-z])", re.I | re.M)

In [ ]:
# Step 3: MD&A = the LONGEST span from an Item 7 heading to the next Item 7A/8

# Every filing has at least two Item 7 headings: one in the table of contents
# and one on the real section. Pair each start with the NEAREST following end
# and the table-of-contents pair is ~100 characters while the real one is tens
# of thousands -- so "take the longest" picks the correct body

def getMDA(cleanText):
  starts = [m.start() for m in ITEM7_RE.finditer(cleanText)]
  ends   = [m.start() for m in ITEM7A_RE.finditer(cleanText)]
  if not starts:
    return ""

  spans = []
  for start in starts:
    end = next((position for position in ends if position > start), len(cleanText))
    spans.append((start, end))

  start, end = max(spans, key=lambda span: span[1] - span[0])
  return cleanText[start:end].strip()


def mdaStatus(mdaText, cleanText, formType):
  """One label per filing"""
  if not formType.upper().startswith("10-K"): # why not "==": "startswith" tolerates "10-K405", "10-KSB", "10-K/A" as well
    return "not an annual report (%s)" % (formType or "no form type")
  if not mdaText:
    return "no Item 7 heading"
  if len(mdaText) < 1000:
    # Asset-backed filers answer Item 7 with "Omitted", per checking there are 5 in our dataset
    return "omitted by filer" if re.search(r"\bomitted\b", mdaText, re.I) else "suspiciously short"

  # sanity check for a reasonable MD&A size
  share = len(mdaText) / len(cleanText)
  if share < 0.01:
    return "suspiciously short"
  if share > 0.60:
    return "suspiciously long"
  return "valid"

In [ ]:
# sanity check on a toy example
toc  = "Item 7.\nManagement's Discussion and Analysis\n61\nItem 7A.\nQuantitative\n73\n"
other = "Our risk factors are described below. " * 300 + "\nSee Item 7 for further detail.\n"
body = ("ITEM\n7. MANAGEMENT'S DISCUSSION AND ANALYSIS\n\n"
        + "The following discussion should be read with our financial statements. " * 40)
after = "\nItem 7A. Quantitative and Qualitative Disclosures\n" + "We face interest rate risk. " * 300

miniTenK = toc + other + body + after
mda = getMDA(miniTenK)

print("starts on the body heading, not the contents:", mda.startswith("ITEM\n7. MANAGEMENT'S"))
print("survives the 'ITEM' / '7' line break       :", bool(ITEM7_RE.search(body)))
print("stops before Item 7A                       :", "Item 7A" not in mda)
print("mid-sentence 'See Item 7' not used         :", "risk factors" not in mda)
print("{:,} chars | status: {}".format(len(mda), mdaStatus(mda, miniTenK, "10-K")))
print()

prose = "blah blah Item 7. Management's Discussion and Analysis Item 7A. Quantitative"
print("prose-only reference:", getMDA(prose))
print("an NT 10-K is labelled, not scored:", mdaStatus("", "short notice", "NT 10-K"))

starts on the body heading, not the contents: True
survives the 'ITEM' / '7' line break       : True
stops before Item 7A                       : True
mid-sentence 'See Item 7' not used         : True
2,885 chars | status: valid

prose-only reference: ''
an NT 10-K is labelled, not scored: not an annual report (NT 10-K)


# Building the df

In [ ]:
DOCS = "/content/drive/MyDrive/Fall2026TextMining_me/Lanqing10Ks"
files = sorted(glob.glob(os.path.join(DOCS, "*.txt")))
print(len(files), "files")

100 files


In [ ]:
# pilot test before running all 100
for path in files[:3]:
  rawSubmission = open(path, "r", encoding="utf-8", errors="ignore").read()
  formType  = getFormType(rawSubmission)
  cleanText = cleanSECText(rawSubmission)
  mdaText   = getMDA(cleanText)

  print(os.path.basename(path), "  form type:", formType)
  print("  raw submission : {:>9,} chars".format(len(rawSubmission)))
  print("  filing itself  : {:>9,} chars".format(len(cleanText)))
  print("  MD&A           : {:>9,} chars  ({:.1%})  [{}]".format(
      len(mdaText), len(mdaText) / len(cleanText) if cleanText else 0,
      mdaStatus(mdaText, cleanText, formType)))
  print("  starts:", repr(mdaText[:100]))
  print("  ends  :", repr(mdaText[-100:]))
  print()

100_10-K_903419_2025-03-14.txt   form type: 10-K
  raw submission : 36,983,584 chars
  filing itself  :   778,601 chars
  MD&A           :   110,294 chars  (14.2%)  [valid]
  starts: 'ITEM 7. MANAGEMENT’S DISCUSSION AND ANALYSIS OF FINANCIAL CONDITION AND RESULTS OF OPERATIONS\n\nThe f'
  ends  : 'mmon share. This dividend is payable on April 11, 2025, to stockholders of record on March 14, 2025.'

10_10-K_1860782_2025-03-25.txt   form type: 10-K
  raw submission : 12,100,667 chars
  filing itself  :   642,583 chars
  MD&A           :    69,335 chars  (10.8%)  [valid]
  starts: 'Item 7. Management’s Discussion and Analysis of Financial Condition and Results of Operations\n\nYou s'
  ends  : 'new or revised accounting standards whenever such early adoption is permitted for private companies.'

11_10-K_1860782_2025-03-17.txt   form type: NT 10-K
  raw submission :    33,635 chars
  filing itself  :     4,058 chars
  MD&A           :         0 chars  (0.0%)  [not an annual report (NT 10-K

In [ ]:
rows = []

for i, path in enumerate(files, 1):
  rawSubmission = open(path, "r", encoding="utf-8", errors="ignore").read()
  formType  = getFormType(rawSubmission)
  cleanText = cleanSECText(rawSubmission)
  mdaText   = getMDA(cleanText)
  status    = mdaStatus(mdaText, cleanText, formType)
  if status != "valid":
    mdaText = ""

  docStats = analyzeDocument(cleanText)
  mdaStats = analyzeDocument(mdaText)

  row = {"fileName": os.path.basename(path), "formType": formType}
  row.update(makeColumns(docStats, ""))        # the whole filing
  row.update(makeColumns(mdaStats, "mda_"))    # Item 7 only
  row["cleanText"]    = cleanText
  row["mdaText"]      = mdaText
  row["raw_chars"]    = len(rawSubmission)
  row["clean_chars"]  = len(cleanText)
  row["mda_chars"]    = len(mdaText)
  row["mdaStatus"]    = status
  row["mdaWordShare"] = rate(row["mda_wordCount"], row["wordCount"])
  rows.append(row)

  print("{:>3} {:<34} {:<8} clean {:>8,}  MD&A {:>7,}  share {:.3f}  {}".format(
      i, row["fileName"], formType, row["clean_chars"], row["mda_chars"],
      row["mdaWordShare"], status))

df = pd.DataFrame(rows)
print(df.shape)

  1 100_10-K_903419_2025-03-14.txt     10-K     clean  778,601  MD&A 110,294  share 0.148  valid
  2 10_10-K_1860782_2025-03-25.txt     10-K     clean  642,583  MD&A  69,335  share 0.110  valid
  3 11_10-K_1860782_2025-03-17.txt     NT 10-K  clean    4,058  MD&A       0  share 0.000  not an annual report (NT 10-K)
  4 12_10-K_1890702_2025-03-07.txt     10-K     clean   80,106  MD&A       0  share 0.000  omitted by filer
  5 13_10-K_1948175_2025-03-10.txt     10-K     clean   75,246  MD&A       0  share 0.000  omitted by filer
  6 14_10-K_933972_2025-03-28.txt               clean        0  MD&A       0  share 0.000  not an annual report (no form type)
  7 15_10-K_910638_2025-03-27.txt      10-K     clean  495,696  MD&A  90,727  share 0.193  valid
  8 16_10-K_910638_2025-03-17.txt      NT 10-K  clean    5,900  MD&A       0  share 0.000  not an annual report (NT 10-K)
  9 17_10-K_66740_2025-02-05.txt       10-K     clean  621,076  MD&A  88,196  share 0.157  valid
 10 18_10-K_1650648_2025-

In [ ]:
# sanity check
print(df["mdaStatus"].value_counts().to_string())
print()

notValid = df.loc[df["mdaStatus"] != "valid",
                  ["fileName", "formType", "mdaStatus", "clean_chars", "mda_chars"]]
print("rows without a usable MD&A:", len(notValid))
print(notValid.to_string(index=False))
print()
print(df.loc[df["mdaStatus"] == "valid",
             ["clean_chars", "mda_chars", "mda_wordCount",
              "mda_sentenceCount", "mdaWordShare"]].describe())

mdaStatus
valid                                  82
not an annual report (NT 10-K)         11
omitted by filer                        5
not an annual report (no form type)     1
suspiciously short                      1

rows without a usable MD&A: 18
                      fileName formType                           mdaStatus  clean_chars  mda_chars
11_10-K_1860782_2025-03-17.txt  NT 10-K      not an annual report (NT 10-K)         4058          0
12_10-K_1890702_2025-03-07.txt     10-K                    omitted by filer        80106          0
13_10-K_1948175_2025-03-10.txt     10-K                    omitted by filer        75246          0
 14_10-K_933972_2025-03-28.txt          not an annual report (no form type)            0          0
 16_10-K_910638_2025-03-17.txt  NT 10-K      not an annual report (NT 10-K)         5900          0
19_10-K_1783875_2025-03-31.txt  NT 10-K      not an annual report (NT 10-K)         7073          0
35_10-K_1173313_2025-03-31.txt  NT 10-K      not

In [ ]:
valid = df[df["mdaStatus"] == "valid"]

print(valid[["fileName", "sentenceCount", "mda_sentenceCount",
             "futureSentenceRate", "mda_futureSentenceRate",
             "FOG", "mda_FOG", "mda_future_FOG"]].head(10).to_string(index=False))
print()
print("n =", len(valid), "filings with a real MD&A")
print("mean futureSentenceRate  whole 10-K: {:.4f}".format(valid["futureSentenceRate"].mean()))
print("mean futureSentenceRate  MD&A      : {:.4f}".format(valid["mda_futureSentenceRate"].mean()))
print("mean FOG                 whole 10-K: {:.4f}".format(valid["FOG"].mean()))
print("mean FOG                 MD&A      : {:.4f}".format(valid["mda_FOG"].mean()))

                      fileName  sentenceCount  mda_sentenceCount  futureSentenceRate  mda_futureSentenceRate       FOG   mda_FOG  mda_future_FOG
100_10-K_903419_2025-03-14.txt           2691                427            0.160164                0.070258 25.113513 24.179568       26.542286
10_10-K_1860782_2025-03-25.txt           2696                306            0.302300                0.143791 24.250309 23.755204       24.906295
 15_10-K_910638_2025-03-27.txt           2115                358            0.134279                0.114525 23.465127 24.591997       26.303622
  17_10-K_66740_2025-02-05.txt           2382                403            0.106633                0.052109 23.247095 21.898768       21.475383
18_10-K_1650648_2025-02-28.txt           2533                203            0.355310                0.216749 24.610039 23.738418       27.239150
 1_10-K_1770787_2025-02-13.txt           2214                220            0.297200                0.150000 23.468974 22.298233  

# Pickle dataset

In [ ]:
PICKLE_PATH = "/content/drive/MyDrive/Fall 2026 Text Mining/week 06/Lanqing Jia/dataFrame.pkl"

df.to_pickle(PICKLE_PATH)
df.drop(columns=["cleanText", "mdaText"]).to_csv(
    "/content/drive/MyDrive/Fall 2026 Text Mining/week 06/Lanqing Jia/readability_mda.csv", index=False)

# reload for sanity check
check = pd.read_pickle(PICKLE_PATH)
print(check.shape)
print(len(check.columns), "columns")
for c in check.columns:
  print(" ", c)

(100, 65)
65 columns
  fileName
  formType
  wordCount
  sentenceCount
  complexWordCount
  PosWordCount
  NegWordCount
  LitWordCount
  PosRate
  NegRate
  LitRate
  complexRate
  wordsPerSentence
  syllablesPerWord
  FOG
  future_wordCount
  future_sentenceCount
  future_complexWordCount
  future_PosWordCount
  future_NegWordCount
  future_LitWordCount
  future_PosRate
  future_NegRate
  future_LitRate
  future_complexRate
  future_wordsPerSentence
  future_syllablesPerWord
  future_FOG
  futureSentenceRate
  futureWordRate
  mda_wordCount
  mda_sentenceCount
  mda_complexWordCount
  mda_PosWordCount
  mda_NegWordCount
  mda_LitWordCount
  mda_PosRate
  mda_NegRate
  mda_LitRate
  mda_complexRate
  mda_wordsPerSentence
  mda_syllablesPerWord
  mda_FOG
  mda_future_wordCount
  mda_future_sentenceCount
  mda_future_complexWordCount
  mda_future_PosWordCount
  mda_future_NegWordCount
  mda_future_LitWordCount
  mda_future_PosRate
  mda_future_NegRate
  mda_future_LitRate
  mda_future_co

In [ ]:
df.head()

,fileName,formType,wordCount,sentenceCount,complexWordCount,PosWordCount,NegWordCount,LitWordCount,PosRate,NegRate,...,mda_future_FOG,mda_futureSentenceRate,mda_futureWordRate,cleanText,mdaText,raw_chars,clean_chars,mda_chars,mdaStatus,mdaWordShare
0,100_10-K_903419_2025-03-14.txt,10-K,88892,2691,26446,467,2015,781,0.005254,0.022668,...,26.542286,0.070258,0.084086,00009034192024-01-012024-12-31\niso4217:USD\n0...,ITEM 7. MANAGEMENT’S DISCUSSION AND ANALYSIS O...,36983584,778601,110294,valid,0.148236
1,10_10-K_1860782_2025-03-25.txt,10-K,88307,2696,24612,942,2115,1152,0.010667,0.023951,...,24.906295,0.143791,0.162352,iso4217:USDxbrli:sharesiso4217:USDxbrli:shares...,Item 7. Management’s Discussion and Analysis o...,12100667,642583,69335,valid,0.109787
2,11_10-K_1860782_2025-03-17.txt,NT 10-K,577,12,119,0,2,9,0.000000,0.003466,...,0.000000,0.000000,0.000000,UNITED STATES\nSECURITIES AND EXCHANGE COMMISS...,,33635,4058,0,not an annual report (NT 10-K),0.000000
3,12_10-K_1890702_2025-03-07.txt,10-K,11208,385,3221,18,256,225,0.001606,0.022841,...,0.000000,0.000000,0.000000,UNITED STATES\n\nSECURITIES AND EXCHANGE COMMI...,,16738243,80106,0,omitted by filer,0.000000
4,13_10-K_1948175_2025-03-10.txt,10-K,10338,308,3195,9,89,148,0.000871,0.008609,...,0.000000,0.000000,0.000000,UNITED STATES\n\nSECURITIES AND EXCHANGE COMMI...,,13253371,75246,0,omitted by filer,0.000000
